# YFinanceSQLite

In [1]:
import yfinance as yf
import sqlite3
import pandas as pd
from datetime import datetime, timedelta
import logging
from typing import Literal

In [ ]:
def setup_database(conn):
    """Create tables if they don't exist."""
    cursor = conn.cursor()

    # Daily stock data table
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS stock_data_daily (
            symbol TEXT NOT NULL,
            date DATE NOT NULL,
            open REAL,
            high REAL,
            low REAL,
            close REAL,
            volume INTEGER,
            PRIMARY KEY (symbol, date)
        )
    """
    )

    # Weekly stock data table
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS stock_data_weekly (
            symbol TEXT NOT NULL,
            date DATE NOT NULL,
            open REAL,
            high REAL,
            low REAL,
            close REAL,
            volume INTEGER,
            PRIMARY KEY (symbol, date)
        )
    """
    )

    # Intraday stock data table (15min, 1min, etc.)
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS stock_data_intraday (
            symbol TEXT NOT NULL,
            datetime TIMESTAMP NOT NULL,
            interval TEXT NOT NULL,
            open REAL,
            high REAL,
            low REAL,
            close REAL,
            volume INTEGER,
            PRIMARY KEY (symbol, datetime, interval)
        )
    """
    )

    # Metadata table to track last update times for different intervals
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS update_log (
            symbol TEXT NOT NULL,
            interval TEXT NOT NULL,
            last_updated TIMESTAMP,
            last_datetime TIMESTAMP,
            PRIMARY KEY (symbol, interval)
        )
    """
    )

    # Create indexes for faster queries\
    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS idx_daily_symbol_date 
        ON stock_data_daily (symbol, date)
    """
    )

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS idx_weekly_symbol_date 
        ON stock_data_weekly (symbol, date)
    """
    )

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS idx_intraday_symbol_datetime 
        ON stock_data_intraday (symbol, datetime)
    """
    )

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS idx_intraday_symbol_interval 
        ON stock_data_intraday (symbol, interval, datetime)
    """
    )

    conn.commit()

In [3]:
def get_last_datetime(conn, symbol, interval):
    """Get the last datetime we have data for a symbol and interval."""
    cursor = conn.cursor()
    cursor.execute(
        """
        SELECT last_datetime FROM update_log 
        WHERE symbol = ? AND interval = ?
    """,
        (symbol, interval),
    )

    result = cursor.fetchone()
    return result[0] if result else None

In [4]:
def _is_daily_interval(interval):
    """Check if interval is daily."""
    return interval == "1d"


def _is_weekly_interval(interval):
    """Check if interval is weekly."""
    return interval == "1wk"

In [5]:
def update_symbol_data(
    conn, symbol, interval="1d", period="1y", force_full_update=False
):
    """
    Download and store data for a symbol with incremental updates.

    Args:
        symbol: Stock symbol (e.g., 'AAPL')
        interval: Data interval ('1m','2m','5m','15m','30m','60m',
                  '90m','1h','1d','1wk','1mo','3mo')
        period: Period to download if no existing data
        force_full_update: If True, re-download all data
    """
    print(f"Processing {symbol} ({interval})...")

    try:
        is_daily  = _is_daily_interval(interval)
        is_weekly = _is_weekly_interval(interval)
        is_eod    = is_daily or is_weekly  # date-only tables

        if is_daily:
            table_name = "stock_data_daily"
        elif is_weekly:
            table_name = "stock_data_weekly"
        else:
            table_name = "stock_data_intraday"

        def _upsert(df):
            """Insert rows, replacing on PRIMARY KEY conflict."""
            cols = list(df.columns)
            placeholders = ", ".join(["?"] * len(cols))
            col_list = ", ".join(cols)
            sql = (
                f"INSERT OR REPLACE INTO {table_name} ({col_list}) "
                f"VALUES ({placeholders})"
            )
            conn.executemany(sql, df.itertuples(index=False, name=None))

        if not force_full_update:
            last_datetime = get_last_datetime(conn, symbol, interval)
            if last_datetime:
                if is_eod:
                    lookback_days = 5 if is_daily else 14
                    start_date = (
                        pd.to_datetime(last_datetime).date() - timedelta(days=lookback_days)
                    ).strftime("%Y-%m-%d")
                    print(f"  Incremental update from {start_date}")
                    ticker = yf.Ticker(symbol)
                    data = ticker.history(start=start_date, interval=interval)
                else:
                    print(f"  Incremental update from {last_datetime}")
                    ticker = yf.Ticker(symbol)
                    data = ticker.history(period="5d", interval=interval)
                    if not data.empty:
                        last_dt = pd.to_datetime(last_datetime)
                        if last_dt.tz is None and data.index.tz is not None:
                            last_dt = last_dt.tz_localize(data.index.tz)
                        elif last_dt.tz is not None and data.index.tz is None:
                            last_dt = last_dt.tz_localize(None)
                        data = data[data.index > last_dt]
            else:
                print(f"  Full download ({period})")
                ticker = yf.Ticker(symbol)
                data = ticker.history(period=period, interval=interval)
        else:
            print(f"  Force full update ({period})")
            ticker = yf.Ticker(symbol)
            data = ticker.history(period=period, interval=interval)
            if is_daily:
                conn.execute("DELETE FROM stock_data_daily WHERE symbol = ?", (symbol,))
            elif is_weekly:
                conn.execute("DELETE FROM stock_data_weekly WHERE symbol = ?", (symbol,))
            else:
                conn.execute(
                    "DELETE FROM stock_data_intraday WHERE symbol = ? AND interval = ?",
                    (symbol, interval),
                )

        if data.empty:
            print(f"  No new data available for {symbol} ({interval})")
            return False

        data.reset_index(inplace=True)
        data["symbol"] = symbol

        if is_eod:
            data["Date"] = data["Date"].dt.date
            data = data.rename(columns={
                "Date": "date", "Open": "open", "High": "high",
                "Low": "low", "Close": "close", "Volume": "volume",
            })
            data = data[["symbol", "date", "open", "high", "low", "close", "volume"]]
            last_point = str(data["date"].max())
            _upsert(data)
        else:
            data["interval"] = interval
            data = data.rename(columns={
                "Datetime": "datetime", "Open": "open", "High": "high",
                "Low": "low", "Close": "close", "Volume": "volume",
            })
            data["datetime"] = (
                data["datetime"].dt.tz_convert("UTC").dt.strftime("%Y-%m-%dT%H:%M:%S%z")
            )
            data = data[["symbol", "datetime", "interval", "open", "high", "low", "close", "volume"]]
            last_point = data["datetime"].max()
            _upsert(data)

        cursor = conn.cursor()
        cursor.execute(
            """
            INSERT OR REPLACE INTO update_log (symbol, interval, last_updated, last_datetime)
            VALUES (?, ?, ?, ?)
        """,
            (symbol, interval, datetime.now().isoformat(), last_point),
        )
        conn.commit()
        print(f"  Successfully stored {len(data)} records for {symbol} ({interval})")
        return True

    except Exception as e:
        print(f"  Error updating {symbol} ({interval}): {str(e)}")
        conn.rollback()
        return False

In [6]:
def update_multiple_symbols(
    conn, symbols, interval="1d", period="1y", force_full_update=False
):
    """Update data for multiple symbols."""
    successful = []
    failed = []

    for symbol in symbols:
        if update_symbol_data(conn, symbol, interval, period, force_full_update):
            successful.append(symbol)
        else:
            failed.append(symbol)

    print(f"\nSummary for {interval}:")
    print(f"  Successful: {len(successful)} symbols")
    print(f"  Failed: {len(failed)} symbols")
    if failed:
        print(f"  Failed symbols: {', '.join(failed)}")

In [7]:
def get_data(conn, symbol, interval="1d", start_date=None, end_date=None):
    """Retrieve data for a symbol from the database."""
    is_daily  = _is_daily_interval(interval)
    is_weekly = _is_weekly_interval(interval)

    if is_daily:
        table, date_col = "stock_data_daily", "date"
        query, params   = f"SELECT * FROM {table} WHERE symbol = ?", [symbol]
    elif is_weekly:
        table, date_col = "stock_data_weekly", "date"
        query, params   = f"SELECT * FROM {table} WHERE symbol = ?", [symbol]
    else:
        table, date_col = "stock_data_intraday", "datetime"
        query           = f"SELECT * FROM {table} WHERE symbol = ? AND interval = ?"
        params          = [symbol, interval]

    if start_date:
        query += f" AND {date_col} >= ?"
        params.append(start_date)
    if end_date:
        query += f" AND {date_col} <= ?"
        params.append(end_date)
    query += f" ORDER BY {date_col}"

    df = pd.read_sql_query(query, conn, params=params)
    if not df.empty:
        if is_daily or is_weekly:
            df[date_col] = pd.to_datetime(df[date_col], utc=False)
        else:
            df[date_col] = pd.to_datetime(df[date_col], utc=True)
        df.set_index(date_col, inplace=True)
    return df

In [8]:
def get_available_symbols(conn, interval="1d"):
    """Get list of all symbols in the database for a specific interval."""
    is_daily  = _is_daily_interval(interval)
    is_weekly = _is_weekly_interval(interval)
    cursor = conn.cursor()
    if is_daily:
        cursor.execute("SELECT DISTINCT symbol FROM stock_data_daily ORDER BY symbol")
    elif is_weekly:
        cursor.execute("SELECT DISTINCT symbol FROM stock_data_weekly ORDER BY symbol")
    else:
        cursor.execute(
            "SELECT DISTINCT symbol FROM stock_data_intraday WHERE interval = ? ORDER BY symbol",
            (interval,),
        )
    return [row[0] for row in cursor.fetchall()]

In [9]:
def get_data_info(conn):
    """Get summary information about stored data."""
    cursor = conn.cursor()

    cursor.execute("""
        SELECT d.symbol, '1d' AS interval, COUNT(*) AS record_count,
               MIN(date) AS first_date, MAX(date) AS last_date, l.last_updated
        FROM stock_data_daily AS d
        LEFT JOIN update_log AS l ON d.symbol = l.symbol AND l.interval = '1d'
        GROUP BY d.symbol
    """)
    daily_data = cursor.fetchall()

    cursor.execute("""
        SELECT w.symbol, '1wk' AS interval, COUNT(*) AS record_count,
               MIN(date) AS first_date, MAX(date) AS last_date, l.last_updated
        FROM stock_data_weekly AS w
        LEFT JOIN update_log AS l ON w.symbol = l.symbol AND l.interval = '1wk'
        GROUP BY w.symbol
    """)
    weekly_data = cursor.fetchall()

    cursor.execute("""
        SELECT symbol, interval, COUNT(*) AS record_count,
               MIN(datetime) AS first_date, MAX(datetime) AS last_date, last_updated
        FROM stock_data_intraday
        LEFT JOIN update_log USING (symbol, interval)
        GROUP BY symbol, interval
    """)
    intraday_data = cursor.fetchall()

    all_data = daily_data + weekly_data + intraday_data
    columns  = ["Symbol", "Interval", "Records", "First Date", "Last Date", "Last Updated"]
    return pd.DataFrame(all_data, columns=columns).sort_values(["Symbol", "Interval"])

In [10]:
def cleanup_old_intraday_data(conn, days_to_keep=30):
    """Remove intraday data older than specified days."""
    cutoff_date = datetime.now() - timedelta(days=days_to_keep)
    cursor = conn.cursor()

    cursor.execute(
        """
        DELETE FROM stock_data_intraday 
        WHERE datetime < ?
    """,
        (cutoff_date,),
    )

    deleted_count = cursor.rowcount
    conn.commit()

    print(f"Deleted {deleted_count} intraday records older than {days_to_keep} days")
    return deleted_count

In [11]:
def close(conn):
    """Close the database connection."""
    conn.close()

## Examples

In [12]:
db_path = "stocks.db"
conn = sqlite3.connect(db_path)
setup_database(conn)

In [13]:
# Define symbols to track
# symbols_1d = ["^GSPC", "^VIX9D", "^VIX", "^VIX3M", "^VIX6M", "^VIX1Y", "ES=F"]
symbols_1d = ["^SPX"]

# Update daily data
print("Updating daily data...")
update_multiple_symbols(conn, symbols_1d, interval="1d", period="1y")

# Example: Get daily data
print(f"\nSample daily data for SPX:")
daily_data = get_data(conn, "^SPX", interval="1d")
print(daily_data.tail())

Updating daily data...
Processing ^SPX (1d)...
  Full download (1y)
  Successfully stored 251 records for ^SPX (1d)

Summary for 1d:
  Successful: 1 symbols
  Failed: 0 symbols

Sample daily data for SPX:
Empty DataFrame
Columns: [symbol, date, open, high, low, close, volume]
Index: []


In [ ]:
# ── Weekly data example ──────────────────────────────────────────
symbols_1wk = ["^SPX"]

print("Updating weekly data...")
update_multiple_symbols(conn, symbols_1wk, interval="1wk", period="5y")

print("\nSample weekly data for ^SPX:")
weekly_data = get_data(conn, "^SPX", interval="1wk")
print(weekly_data.tail())

In [15]:
# symbols_15m = ["^GSPC", "^VIX", "ES=F"]
symbols_15m = ["^SPX"]

# Update 15-minute data (note: limited historical data available)
print("\nUpdating 15-minute data...")
update_multiple_symbols(conn, symbols_15m, interval="15m", period="7d")

# Example: Get 15-minute data
print(f"\nSample 15-minute data for SPX:")
intraday_data = get_data(conn, "^SPX", interval="15m")
print(intraday_data.tail())


Updating 15-minute data...
Processing ^SPX (15m)...
  Incremental update from 2026-04-01T19:45:00+0000
  No new data available for ^SPX (15m)

Summary for 15m:
  Successful: 0 symbols
  Failed: 1 symbols
  Failed symbols: ^SPX

Sample 15-minute data for SPX:
                          symbol interval         open         high  \
datetime                                                              
2026-04-01 18:45:00+00:00   ^SPX      15m  6572.720215  6573.810059   
2026-04-01 19:00:00+00:00   ^SPX      15m  6568.080078  6577.959961   
2026-04-01 19:15:00+00:00   ^SPX      15m  6576.270020  6587.600098   
2026-04-01 19:30:00+00:00   ^SPX      15m  6585.720215  6591.370117   
2026-04-01 19:45:00+00:00   ^SPX      15m  6586.299805  6587.609863   

                                   low        close     volume  
datetime                                                        
2026-04-01 18:45:00+00:00  6566.200195  6568.069824   70898710  
2026-04-01 19:00:00+00:00  6567.290039  6576.31

In [16]:
# Show summary of stored data
print("\nData Summary:")
print(get_data_info(conn))


Data Summary:
  Symbol Interval  Records                First Date  \
1   ^SPX      15m      182  2026-03-24T13:30:00+0000   
0   ^SPX    daily      251                2025-04-02   

                  Last Date                Last Updated  
1  2026-04-01T19:45:00+0000  2026-04-02 12:55:09.294888  
0                2026-04-01  2026-04-02 12:54:46.114050  


In [17]:
# Clean up old intraday data (optional)
cleanup_old_intraday_data(conn, days_to_keep=7)

Deleted 52 intraday records older than 7 days


52

In [18]:
# Close the database connection
close(conn)